## Program the FPGA

In [ ]:
#programming FPGA

import chipwhisperer as cw
try: 
    scope.dis()
    target.dis()
except:
    print("")
scope = cw.scope()
#bitstream location
target = cw.target(scope, cw.targets.CW305, bsfile=r'path\to\cw305_top.bit',force=True)
print("done");

In [ ]:
#Clock and voltage
target.vccint_set(1.0)

# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 100 MHz:
target.pll.pll_outfreq_set(100E6, 1)

## CMDs

In [ ]:
#Define commands
import serial
import time

# TDC Commands
CMD_GET_TDC_VAL              = b'\x29'
CMD_SET_TDC_VAL              = b'\x2A'
CMD_SET_TDC_ID               = b'\x2B'
CMD_GET_TDC_ID               = b'\x2C'
CMD_SET_TDC_MEAS_CYCLE       = b'\x2D'
CMD_GET_TDC_MEAS_CYCLE       = b'\x2E'
CMD_GET_MEAS_TDC             = b'\x2F'
CMD_TRIG_MEAS_TDC            = b'\x30'
CMD_GET_COARSE_TUNE_TDC      = b'\x31'
CMD_SET_COARSE_TUNE_TDC      = b'\x32'
CMD_GET_FINE_TUNE_TDC        = b'\x33'
CMD_SET_FINE_TUNE_TDC        = b'\x34'
CMD_SET_TDC_FREQ_INT         = b'\x35'
CMD_GET_TDC_FREQ_INT         = b'\x36'
CMD_SET_TDC_FREQ_FRAC        = b'\x37'
CMD_GET_TDC_FREQ_FRAC        = b'\x38'
CMD_SET_TDC_READ_ADDR        = b'\x39'
CMD_GET_TDC_READ_ADDR        = b'\x3A'


# TDC TEST Commands
CMD_SET_CAL_TDC_TEST         = b'\x41'
CMD_GET_MEAS_TDC_TEST        = b'\x42'


def open_serial_device(serial_dev: str, baudrate: int, timeout: float) -> serial.Serial:
    ser: serial.Serial = serial.Serial(serial_dev, baudrate, timeout=timeout)
    return ser

def close_serial_device(ser: serial.Serial):
    ser.close()


# TDC Functions
def get_tdc_id(ser: serial.Serial) -> bytes:
    ser.write(CMD_GET_TDC_ID)
    response = ser.read(1)
    return response

def set_tdc_id(ser: serial.Serial, value: bytes):
    ser.write(CMD_SET_TDC_ID)
    ser.write(value)
    
def set_tdc_meas_cycle(ser: serial.Serial, value: int):
    ser.write(CMD_SET_TDC_MEAS_CYCLE)
    ser.write(value.to_bytes(2, 'big'))  
    
def get_tdc_meas_cycle(ser: serial.Serial) -> int:
    ser.write(CMD_GET_TDC_MEAS_CYCLE)
    response = ser.read(2)
    return int.from_bytes(response, 'big')

def trigger_tdc_measure(ser: serial.Serial):
    ser.write(CMD_TRIG_MEAS_TDC)

def get_tdc_meas_result(ser: serial.Serial, data_width : int) -> bytes:
    ser.write(CMD_GET_MEAS_TDC)
    response = ser.read(int(data_width/8))  
    return response

def set_tdc_coarse_tune(ser: serial.Serial, value: bytes):
    ser.write(CMD_SET_COARSE_TUNE_TDC)
    ser.write(value)  

    
def set_tdc_read_address(ser: serial.Serial, value: bytes):
    ser.write(CMD_SET_TDC_READ_ADDR)
    ser.write(value)

def get_tdc_read_address(ser: serial.Serial) -> bytes:
    ser.write(CMD_GET_TDC_READ_ADDR)
    response = ser.read(2)
    return response 


def set_tdc_test_cal(ser: serial.Serial, value: bytes):
    ser.write(CMD_SET_CAL_TDC_TEST )
    ser.write(value)
    
def get_tdc_test_meas(ser: serial.Serial) -> bytes:
    ser.write(CMD_GET_MEAS_TDC_TEST)
    response = ser.read(16)
    return response

def get_tdc_measurement_cycle(ser: serial.Serial):
    cycle_length = get_tdc_meas_cycle(ser)
    print("TDC Measurement Cycle is: {}".format(cycle_length))

# GPIO  fucntions
def set_gpio(ser: serial.Serial):
    ser.write(CMD_SET_GPIO)

def unset_gpio(ser: serial.Serial):
    ser.write(CMD_UNSET_GPIO)
    

## Serial Connection

In [ ]:
import sys

import numpy as np
import time
from tqdm import tnrange
try: 
    close_serial_device(fpga_ser)
except: 
    print("")

fpga_ser = open_serial_device(serial_dev="COM15", baudrate=115200, timeout=1)  #change COM15


## TDC

In [ ]:
def trigger_and_get_tdc_measurement(ser: serial.Serial, read_address: bytes, data_width : int, tdc_id ):
    # Set TDC ID
    
    set_tdc_id(ser,  tdc_id)
    time.sleep(0.01)
    print ("Current TDC ID is: ", get_tdc_id(ser))
    # Trigger the TDC measurement
    trigger_tdc_measure(ser)
    
    # Wait for the measurement to complete (adjust sleep time based on TDC response)
    time.sleep(0.1)
    
    # Get the measurement cycle length (number of entries to read)
    cycle_length = get_tdc_meas_cycle(ser) 
    time.sleep(0.01)
    
    # Update the TDC read address
    set_tdc_read_address(ser, read_address)
    
    # Read each data entry corresponding to the measurement cycle length
    results = []
    address_int = int.from_bytes(read_address, byteorder='big')
    for i in range(cycle_length):
        # Read the measurement data from the TDC's memory
        result = get_tdc_meas_result(ser, data_width)
        time.sleep(.2)
        results.append(result)
        

        # Increment the integer
        address_int += 1
    
        # Convert the integer back to a 2-byte bytes object
        # Ensure it stays 2 bytes by using .to_bytes(2, 'big')
        address_byte = address_int.to_bytes(2, byteorder='big')
        
        
        set_tdc_read_address(ser, address_byte)
        time.sleep(0.01)
    # Print and return the list of measurement results
    print(f"TDC Measurement Results (Cycle Length: {cycle_length}):")
    for i, result in enumerate(results):
        print(f"Entry {(int.from_bytes(read_address, byteorder='big')+i).to_bytes(2, byteorder='big')}: {result}")
    
    return results



In [ ]:
#Calibration Function
# def count_ones_in_result(result):
#     """Helper function to count the number of '1's in a byte sequence"""
#     return sum(bin(byte).count('1') for byte in result)


def count_ones_in_result(result):

    bits = ''.join(f'{byte:08b}' for byte in result)

    # If all bits are 1, count all of them
    if all(b == '1' for b in bits):
        return len(bits)
    
    # count leading 1s
    left_count = 0
    for b in bits:
        if b == '1':
            left_count += 1
        else:
            break

    # count trailing 1s
    right_count = 0
    for b in reversed(bits):
        if b == '1':
            right_count += 1
        else:
            break

    # If 1s appear on both sides, ignore it
    if left_count > 0 and right_count > 0:
        return 0
    

    return max(left_count, right_count)


def calibrate_tdc_coarse_tune(fpga_ser, read_address, coarse_size,  tdc_id , sensor_width=64):
    """
    Perform binary search to find the optimal coarse tune setting by triggering TDC measurements
    and checking if there is any '1' in the response.

    :param fpga_ser: The serial object for FPGA communication.
    :param read_address: The starting read address for TDC measurement results.
    :param coarse_min: The minimum value for the coarse tune range.
    :param coarse_max: The maximum value for the coarse tune range.
    :return: The optimal coarse tune value that produced a valid measurement.
    """
    value = 0
    target_ones = sensor_width/2  # Half of 128 bits
    closest_index = None
    closest_byte_value = None
    closest_diff = float('inf')  # Initialize with infinity so any diff will be closer

    # Loop to generate a 32-bit value, setting one bit at a time from right to left
    print("coarse_size: ", coarse_size)
    print(f"----------------------------------------------{tdc_id}----------------------------------------------")
    for i in range(coarse_size):
        # Set the i-th bit (from the right) to 1 using bit-shifting
        value = value | (1 << (31 - i))

        # Convert the 32-bit value to a 4-byte (32-bit) bytes object in big-endian format
        byte_value = value.to_bytes(4, byteorder='big')

        # Print the result in bytes format
        print(f"------------------------------Iteration {i+1}: {byte_value}------------------------------")
        
        # Send the value to the TDC for testing
        set_tdc_coarse_tune(fpga_ser, byte_value)
        time.sleep(0.1)
        
        # Get the measurement results (128-bit, assumed to be a list of 16 bytes)
        measurement_results = trigger_and_get_tdc_measurement(fpga_ser, read_address,  sensor_width, tdc_id)
        print(f"Measurement Results (64-bit): {measurement_results}")
        
        # Count the number of '1's in the result
#         print(len(measurement_results))
        ones_count = count_ones_in_result(measurement_results[0])
        print(f"Number of '1's in the result: {ones_count}")

        # Calculate the difference between current '1's count and target (64)
        diff = abs(ones_count - target_ones)

        # If this result is closer to 64 '1's than any previous result, update
        if diff < closest_diff:
            closest_diff = diff
            closest_index = i + 1  # Store 1-based index
            closest_byte_value = byte_value

    # Return the index and the byte value that had the closest number of '1's to 64
    print(f"Closest result found at index {closest_index} with byte value {closest_byte_value}")
    print(f"Closest diff found at index {closest_index} with diff value {closest_diff}")

    return closest_byte_value


def binary_search_tdc_cal_test(coarse_size, sensor_size)->bytes:
    value = 0
    target_ones = sensor_size/2  # Half of 128 bits
    closest_index = None
    closest_byte_value = None
    closest_diff = float('inf')  # Initialize with infinity so any diff will be closer

    # Loop to generate a 32-bit value, setting one bit at a time from right to left
    for i in range(coarse_size):
        # Set the i-th bit (from the right) to 1 using bit-shifting
        value = value | (1 << (31 - i))

        # Convert the 32-bit value to a 4-byte (32-bit) bytes object in big-endian format
        byte_value = value.to_bytes(4, byteorder='big')

        # Print the result in bytes format
#         print(f"Iteration {i+1}: {byte_value}")
        
        # Send the value to the TDC for testing
        set_tdc_test_cal(fpga_ser, byte_value)
        time.sleep(0.1)
        
        # Get the measurement results (128-bit, assumed to be a list of 16 bytes)
        measurement_results = get_tdc_test_meas(fpga_ser)
#         print(f"Measurement Results (64-bit): {measurement_results}")
        
        # Count the number of '1's in the result
        ones_count = count_ones_in_result(measurement_results)
#         print(f"Number of '1's in the result: {ones_count}")

        # Calculate the difference between current '1's count and target (64)
        diff = abs(ones_count - target_ones)

        # If this result is closer to 64 '1's than any previous result, update
        if diff < closest_diff:
            closest_diff = diff
            closest_index = i + 1  # Store 1-based index
            closest_byte_value = byte_value

    # Return the index and the byte value that had the closest number of '1's to 64
    print(f"Closest result found at index {closest_index} with byte value {closest_byte_value}")
    return closest_byte_value




In [ ]:
set_tdc_meas_cycle(fpga_ser,1)

get_tdc_measurement_cycle(fpga_ser)

optimal_coarse_tune_tdc = []
Number_of_tdc = 79
tdc_ids = [bytes([i]) for i in range(Number_of_tdc)]

print(len(tdc_ids))
for i in range(len(tdc_ids)):
    sensor_width = 64
    coarse_size  = 32
    optimal_byte_value = binary_search_tdc_cal_test(coarse_size,sensor_width)
    read_address = b'\x00\x00'
    set_tdc_read_address(fpga_ser,read_address)
    print("The raed Address is: ", get_tdc_read_address(fpga_ser))
    optimal_coarse_tune_tdc.append(calibrate_tdc_coarse_tune(fpga_ser, read_address, coarse_size, tdc_ids[i], sensor_width))

In [ ]:
import matplotlib.pyplot as plt
import time
import pyvisa
from tqdm import tnrange
import numpy as np
import time
# from Crypto.Cipher import AES
import random as rd
import os
import shutil

sensor_width = 64
read_address = b'\x00\x00'
number_of_traces = 1   #number of trces. increase it for more accuracy. (ex: 1000)

y_values = []

parent_dir = r'path\to\output\folder'
directory = "TDC_results"

# Safe way using os.path.join
path = os.path.join(parent_dir, directory)
print(path)
path_dir = path
# Ampl = 20;
N = 100    #Number of TDC measurement cycles. A larger value improves measurement accuracy.  (ex: 1000)

# #######################################################################################
# # # signal generator (n9310a) setup
# rm = pyvisa.ResourceManager()
# instruments = rm.list_resources()
# print(f"Connected instruments: {instruments}")
# n9310a_address = instruments[4]
# n9310a = rm.open_resource(n9310a_address)
# idn = n9310a.query('*IDN?')

# print(f'Device ID: {idn}')
# n9310a.write(":AMPLITUDE:CW "+str(Ampl)+"dBm")
# #######################################################################################

for item in range(number_of_traces):
    y_values = []
#     n9310a.write(":FREQ:CW " + str(item) + "Hz")
    time.sleep(1);
    for tdc_idx in range(Number_of_tdc):
        # Set coarse tune and measurement cycle
        set_tdc_coarse_tune(fpga_ser, optimal_coarse_tune_tdc[tdc_idx])
        time.sleep(.1)
        set_tdc_meas_cycle(fpga_ser, N)
        time.sleep(.1)
        tdc_id = tdc_ids[tdc_idx]
        time.sleep(.1)
        # Get measurement results
        measurement_results = trigger_and_get_tdc_measurement(fpga_ser, read_address, sensor_width, tdc_id)
        time.sleep(.1)
        # Collect data for the current TDC (count of ones)
        y_values.append([count_ones_in_result(measurement_results[i]) for i in range(len(measurement_results))])
        time.sleep(.1)
    file = open(path_dir+"TDCs_"+"Freq_" + str(item) + "dbm.trc", "w+")
    content = str(y_values)
    file.write(content)
    file.close()

# n9310a.close()

x = list(range(len(y_values[0])))




In [ ]:
import os
import json
import numpy as np

def read_trc_file(file_path):
    """Read a .trc file and return the list of TDC values."""
    with open(file_path, 'r') as file:
        data = json.loads(file.read())  # Assuming the .trc file content is in JSON format
    return data

def calculate_average(data):
    """Calculate the average for each TDC."""
    averages = [np.mean(tdc) for tdc in data]
    return averages

def process_trc_files(directory):
    """Process all .trc files in the given directory."""
    all_averages = []
    for filename in os.listdir(directory):
        if filename.endswith('.trc'):
            file_path = os.path.join(directory, filename)
            data = read_trc_file(file_path)
            averages = calculate_average(data)
            all_averages.append(averages)
            print(f"Averages for {filename}: {averages}")
    return all_averages

# Directory containing your .trc files
trc_directory = parent_dir
# directory = "TDC_results"


# Calculate and print averages for all files
averages = process_trc_files(trc_directory)

# #global average across all files for each TDC:
# global_average = np.mean(averages, axis=0)
# print(f"Global average across all files: {global_average}")
